# SBO ML — Kaggle training

Notebook độc lập: mã trong `training/` đã được nhúng vào cell khởi tạo. Chỉ cần cung cấp CSV gốc
`dataset_labeled_sbo_rich_v6_context_clean.csv` (hoặc tar.gz chứa file này) qua dữ liệu đầu vào Kaggle.
Không dùng CSV 200 dòng của demo để báo cáo chất lượng.

Luồng mới tách nhóm train/meta/select/test **trước** khi chọn đặc trưng. Feature ranking, audit và
chọn 160/260 feature chỉ nhìn tập train; các bước tune threshold/ensemble dùng validation.
Số liệu mới sẽ khác log cũ và chưa được tạo sẵn trong notebook này.


In [ ]:
from pathlib import Path
import sys, json, shutil, tarfile, subprocess, importlib.metadata, zipfile
from datetime import datetime, timezone
from uuid import uuid4

INPUT_ROOT = Path('/kaggle/input')
WORK = Path('/kaggle/working/SBO_ML') / (datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S') + '_' + uuid4().hex[:6])
WORK.mkdir(parents=True, exist_ok=False)
FEATURE_COUNT = 260  # 260 hoặc 160; dùng validation để quyết định giữa các cấu hình.
N_JOBS = 4
SEED = 42
CSV_NAME = 'dataset_labeled_sbo_rich_v6_context_clean.csv'
INPUT_FILE = None  # Điền Path('/kaggle/input/.../file.csv') nếu có nhiều bộ dữ liệu.
print('Output directory:', WORK)


## Khởi tạo mã nguồn
Cell dưới được sinh từ mã trong repository; không chứa log hoặc kết quả train cũ.


In [ ]:
BUNDLED_SOURCES = {'training/__init__.py': '"""Training and feature analysis recovered from the supplied Kaggle session."""\n', 'training/analyze_v6_top260_components.py': '#!/usr/bin/env python3\n# -*- coding: utf-8 -*-\n\nimport argparse\nfrom pathlib import Path\nimport re\n\nimport numpy as np\nimport pandas as pd\nimport matplotlib.pyplot as plt\n\n\nID_COLS = {\n    "binary", "binary_norm", "program_name", "json_file", "source_core",\n    "build_config", "function", "name_norm", "entry", "schema",\n    "merge_stage", "candidate_reasons", "label",\n}\n\n\ndef feature_group(c):\n    c = str(c)\n    cl = c.lower()\n\n    if c.startswith(("pcode_", "mnemonic_")):\n        return "opcode_like"\n\n    if c.startswith(("local_buffers_", "v6_aliases_")):\n        return "raw_buffer_alias"\n\n    if c.startswith("sinks_v6_"):\n        return "v6_sink"\n\n    if c.startswith(("v6_summary_", "v6_loop_summary_", "v6_function_role_")):\n        return "v6_summary"\n\n    if c.startswith(("sink_summary_", "buffer_summary_", "v4_summary_")):\n        return "sink_buffer_summary"\n\n    if c.startswith(("stack_", "cfg_", "summary_", "callgraph_")):\n        return "program_static"\n\n    if "risk" in cl or "safe" in cl or "guard" in cl or "overflow" in cl:\n        return "risk_safe"\n\n    if "dst" in cl or "src" in cl or "source" in cl or "size" in cl or "sink" in cl:\n        return "dataflow_sink_size"\n\n    return "other"\n\n\ndef numeric_feature_cols(df):\n    out = []\n\n    for c in df.columns:\n        if c in ID_COLS:\n            continue\n\n        s = pd.to_numeric(df[c], errors="coerce")\n        if s.notna().sum() == 0:\n            continue\n\n        s = s.replace([np.inf, -np.inf], np.nan).fillna(0)\n        if s.nunique(dropna=False) <= 1:\n            continue\n\n        out.append(c)\n\n    return out\n\n\ndef calc_feature_stats(df, cols):\n    y = df["label"].astype(int).values\n    rows = []\n\n    for c in cols:\n        s = pd.to_numeric(df[c], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)\n\n        s0 = s[y == 0]\n        s1 = s[y == 1]\n\n        std = float(s.std())\n        sep = 0.0 if std == 0 else abs(float(s1.mean()) - float(s0.mean())) / std\n\n        rows.append({\n            "feature": c,\n            "group": feature_group(c),\n            "nonzero_ratio": float((s != 0).mean()),\n            "unique_count": int(s.nunique(dropna=False)),\n            "mean_label0": float(s0.mean()),\n            "mean_label1": float(s1.mean()),\n            "abs_mean_diff": abs(float(s1.mean()) - float(s0.mean())),\n            "class_separation": sep,\n        })\n\n    return pd.DataFrame(rows)\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n\n    ap.add_argument("--data", required=True)\n    ap.add_argument("--rank", required=True)\n    ap.add_argument("--case", default="score_corr_top260")\n    ap.add_argument("--out-dir", default="outputs/reports/top260_feature_audit")\n\n    args = ap.parse_args()\n\n    data_p = Path(args.data)\n    rank_p = Path(args.rank)\n    out_dir = Path(args.out_dir)\n    out_dir.mkdir(parents=True, exist_ok=True)\n\n    df = pd.read_csv(data_p, low_memory=False)\n    rank = pd.read_csv(rank_p)\n\n    selected = numeric_feature_cols(df)\n\n    rank_cols = [c for c in ["feature", "score", "group", "corr_pruned_order"] if c in rank.columns]\n    rank_small = rank[rank_cols].copy()\n\n    stats = calc_feature_stats(df, selected)\n\n    if "feature" in rank_small.columns:\n        audit = stats.merge(rank_small, on="feature", how="left", suffixes=("", "_rank"))\n    else:\n        audit = stats.copy()\n\n    if "score" not in audit.columns:\n        audit["score"] = audit["class_separation"]\n\n    audit = audit.sort_values(["score", "class_separation"], ascending=False)\n\n    comp = (\n        audit.groupby("group")\n        .agg(\n            feature_count=("feature", "count"),\n            mean_score=("score", "mean"),\n            max_score=("score", "max"),\n            mean_separation=("class_separation", "mean"),\n            mean_nonzero=("nonzero_ratio", "mean"),\n        )\n        .reset_index()\n        .sort_values(["feature_count", "mean_score"], ascending=False)\n    )\n\n    audit.to_csv(out_dir / f"{args.case}_feature_audit.csv", index=False)\n    comp.to_csv(out_dir / f"{args.case}_group_composition.csv", index=False)\n\n    top = audit.head(80)\n    top.to_csv(out_dir / f"{args.case}_top80_features.csv", index=False)\n\n    print("========== GROUP COMPOSITION ==========")\n    print(comp.to_string(index=False))\n\n    print("\\n========== TOP 40 FEATURES ==========")\n    print(top[["feature", "group", "score", "class_separation", "nonzero_ratio"]].head(40).to_string(index=False))\n\n    # Plot 1: group counts\n    plt.figure(figsize=(12, 6))\n    plt.bar(comp["group"], comp["feature_count"])\n    plt.title(f"{args.case}: Feature group composition")\n    plt.ylabel("Number of features")\n    plt.xticks(rotation=30, ha="right")\n    plt.tight_layout()\n    plt.savefig(out_dir / f"{args.case}_group_counts.png", dpi=220)\n    plt.close()\n\n    # Plot 2: mean score by group\n    comp2 = comp.sort_values("mean_score", ascending=False)\n\n    plt.figure(figsize=(12, 6))\n    plt.bar(comp2["group"], comp2["mean_score"])\n    plt.title(f"{args.case}: Mean selector score by group")\n    plt.ylabel("Mean score")\n    plt.xticks(rotation=30, ha="right")\n    plt.tight_layout()\n    plt.savefig(out_dir / f"{args.case}_group_mean_score.png", dpi=220)\n    plt.close()\n\n    # Plot 3: top feature scores\n    top30 = audit.head(30).iloc[::-1]\n\n    plt.figure(figsize=(12, 9))\n    plt.barh(top30["feature"], top30["score"])\n    plt.title(f"{args.case}: Top 30 selected features")\n    plt.xlabel("Selector score")\n    plt.tight_layout()\n    plt.savefig(out_dir / f"{args.case}_top30_features.png", dpi=220)\n    plt.close()\n\n    print("\\n[OK] wrote:", out_dir)\n\n\nif __name__ == "__main__":\n    main()\n', 'training/export_feature_inventory_for_review.py': '#!/usr/bin/env python3\n# -*- coding: utf-8 -*-\n\nimport argparse\nimport json\nfrom pathlib import Path\n\nimport joblib\nimport numpy as np\nimport pandas as pd\n\n\nID_COLS = {\n    "binary", "binary_norm", "program_name", "json_file", "source_core",\n    "build_config", "function", "name_norm", "entry", "schema",\n    "merge_stage", "candidate_reasons", "label",\n}\n\n\ndef log(*x):\n    print(*x, flush=True)\n\n\ndef auto_find_data():\n    patterns = [\n        "outputs/features/**/dataset_v6_pruned_score_corr_top260.csv",\n        "outputs/features/final_selected/dataset_security_full_top260.csv",\n        "outputs/features/dataset_labeled_sbo_rich_v6_context_clean.csv",\n        "samples/dataset_v6_pruned_score_corr_top260.csv",\n    ]\n\n    cands = []\n    for pat in patterns:\n        cands.extend(Path(".").glob(pat))\n\n    cands = [p for p in cands if p.exists()]\n\n    if not cands:\n        return None\n\n    def priority(p):\n        s = str(p)\n        if "score_corr_top260" in s:\n            return 0\n        if "final_selected" in s:\n            return 1\n        if "v6_context_clean" in s:\n            return 2\n        return 9\n\n    cands = sorted(cands, key=lambda p: (priority(p), len(str(p))))\n    return cands[0]\n\n\ndef auto_find_model_dir():\n    patterns = [\n        "outputs/models*top260",\n        "outputs/**/models*top260",\n        "models/*top260*",\n        "models/final_model_security_top260",\n    ]\n\n    cands = []\n    for pat in patterns:\n        cands.extend(Path(".").glob(pat))\n\n    cands = [p for p in cands if p.is_dir() and (p / "model_metrics.csv").exists()]\n\n    if not cands:\n        return None\n\n    return sorted(cands, key=lambda p: len(str(p)))[0]\n\n\ndef infer_model_features(model_dir):\n    if not model_dir:\n        return None, None, {}\n\n    model_dir = Path(model_dir)\n    if not model_dir.exists():\n        return None, None, {}\n\n    feature_cols = None\n    source_model = None\n    importances = {}\n\n    for name in [\n        "RandomForest",\n        "DecisionTree",\n        "XGBoost",\n        "LogisticRegression",\n    ]:\n        p = model_dir / f"{name}.joblib"\n        if not p.exists():\n            continue\n\n        try:\n            m = joblib.load(p)\n        except Exception as e:\n            log("[WARN] cannot load", p, e)\n            continue\n\n        if feature_cols is None and hasattr(m, "feature_names_in_"):\n            feature_cols = list(m.feature_names_in_)\n            source_model = name\n\n        if hasattr(m, "feature_importances_") and hasattr(m, "feature_names_in_"):\n            for f, v in zip(m.feature_names_in_, m.feature_importances_):\n                importances[str(f)] = float(v)\n\n    return feature_cols, source_model, importances\n\n\ndef feature_group(c):\n    c = str(c)\n    cl = c.lower()\n\n    if c.startswith(("pcode_", "mnemonic_")):\n        return "opcode_like"\n\n    if c.startswith("sinks_v6_"):\n        return "v6_sink"\n\n    if c.startswith(("sink_summary_", "buffer_summary_", "v4_summary_")):\n        return "sink_buffer_summary"\n\n    if c.startswith(("local_buffers_", "v6_aliases_", "v6_alias_summary_")):\n        return "raw_buffer_alias"\n\n    if c.startswith(("v6_summary_", "v6_loop_summary_", "v6_function_role_")):\n        return "v6_summary"\n\n    if c.startswith("cfg_"):\n        return "cfg_control_flow"\n\n    if c.startswith("stack_"):\n        return "stack_memory"\n\n    if c.startswith("callgraph_"):\n        return "callgraph_context"\n\n    if any(k in cl for k in [\n        "risk", "safe", "guard", "bound", "check", "overflow"\n    ]):\n        return "risk_safe"\n\n    if any(k in cl for k in [\n        "dst", "src", "source", "size", "sink", "line_index"\n    ]):\n        return "dataflow_sink_size"\n\n    if c.startswith("summary_"):\n        return "program_static"\n\n    return "other"\n\n\ndef proxy_flags(c):\n    cl = str(c).lower()\n    flags = []\n\n    patterns = {\n        "line_index": ["line_index"],\n        "stack_offset": ["stack_offset"],\n        "size_proxy": [\n            "instruction_count",\n            "decompiled_line_count",\n            "line_count",\n            "caller_names_count",\n            "callee_names_count",\n        ],\n        "generic_count": [\n            "summary_instruction_count",\n            "callgraph_caller_names_count",\n        ],\n    }\n\n    for name, pats in patterns.items():\n        if any(p in cl for p in pats):\n            flags.append(name)\n\n    return ",".join(flags)\n\n\ndef normalize(s):\n    s = pd.Series(s).replace([np.inf, -np.inf], np.nan).fillna(0.0)\n    mn, mx = float(s.min()), float(s.max())\n    if abs(mx - mn) < 1e-12:\n        return pd.Series(np.zeros(len(s)), index=s.index)\n    return (s - mn) / (mx - mn)\n\n\ndef get_numeric_features(df):\n    cols = []\n\n    for c in df.columns:\n        if c in ID_COLS:\n            continue\n\n        x = pd.to_numeric(df[c], errors="coerce")\n        if x.notna().sum() == 0:\n            continue\n\n        x = x.replace([np.inf, -np.inf], np.nan).fillna(0)\n        if x.nunique(dropna=False) <= 1:\n            continue\n\n        cols.append(c)\n\n    return cols\n\n\ndef compute_audit(df, feature_cols, rf_importance):\n    rows = []\n\n    present = [c for c in feature_cols if c in df.columns]\n\n    X = df[present].apply(pd.to_numeric, errors="coerce")\n    X = X.replace([np.inf, -np.inf], np.nan).fillna(0.0)\n\n    y = None\n    has_label = "label" in df.columns and set(pd.unique(df["label"].dropna())).issubset({0, 1})\n\n    if has_label:\n        y = df["label"].astype(int).values\n        y_mean = y.mean()\n        y_std = y.std() + 1e-12\n\n    mi_map = {}\n\n    if has_label:\n        try:\n            from sklearn.feature_selection import mutual_info_classif\n            mi = mutual_info_classif(\n                X.values,\n                y,\n                random_state=1337,\n                discrete_features=False,\n            )\n            mi_map = {c: float(v) for c, v in zip(present, mi)}\n        except Exception as e:\n            log("[WARN] mutual_info failed:", e)\n\n    for c in present:\n        x = X[c].values.astype(float)\n\n        nonzero = float((x != 0).mean())\n        nunique = int(pd.Series(x).nunique(dropna=False))\n\n        mean_all = float(np.mean(x))\n        std_all = float(np.std(x))\n\n        mean_safe = np.nan\n        mean_vuln = np.nan\n        diff = np.nan\n        separation = np.nan\n        corr = np.nan\n\n        if has_label:\n            x0 = x[y == 0]\n            x1 = x[y == 1]\n\n            mean_safe = float(np.mean(x0)) if len(x0) else 0.0\n            mean_vuln = float(np.mean(x1)) if len(x1) else 0.0\n            diff = mean_vuln - mean_safe\n            separation = abs(diff) / (std_all + 1e-12)\n\n            if std_all > 1e-12:\n                corr = float(np.mean((x - x.mean()) * (y - y_mean)) / ((std_all + 1e-12) * y_std))\n\n        rows.append({\n            "feature": c,\n            "group": feature_group(c),\n            "proxy_flags": proxy_flags(c),\n            "nonzero_rate": nonzero,\n            "nunique": nunique,\n            "mean_all": mean_all,\n            "mean_safe": mean_safe,\n            "mean_vulnerable": mean_vuln,\n            "mean_diff_vuln_minus_safe": diff,\n            "separation": separation,\n            "abs_corr_label": abs(corr) if not pd.isna(corr) else 0.0,\n            "corr_label": corr if not pd.isna(corr) else 0.0,\n            "mutual_info": mi_map.get(c, 0.0),\n            "rf_importance": rf_importance.get(c, 0.0),\n        })\n\n    audit = pd.DataFrame(rows)\n\n    if audit.empty:\n        return audit\n\n    audit["score_corr_n"] = normalize(audit["abs_corr_label"])\n    audit["score_mi_n"] = normalize(audit["mutual_info"])\n    audit["score_sep_n"] = normalize(audit["separation"])\n    audit["score_rf_n"] = normalize(audit["rf_importance"])\n\n    audit["audit_score"] = (\n        0.35 * audit["score_corr_n"]\n        + 0.25 * audit["score_mi_n"]\n        + 0.20 * audit["score_sep_n"]\n        + 0.20 * audit["score_rf_n"]\n    )\n\n    semantic_core = {\n        "v6_sink",\n        "sink_buffer_summary",\n        "risk_safe",\n        "stack_memory",\n        "raw_buffer_alias",\n        "v6_summary",\n    }\n\n    audit["semantic_core"] = audit["group"].isin(semantic_core).astype(int)\n    audit["has_proxy_flag"] = (audit["proxy_flags"].astype(str) != "").astype(int)\n\n    def hint(row):\n        if row["semantic_core"] and row["audit_score"] >= 0.25:\n            return "strong_keep_candidate"\n        if row["semantic_core"] and row["audit_score"] >= 0.12:\n            return "keep_review"\n        if row["has_proxy_flag"] and row["audit_score"] < 0.25:\n            return "proxy_review"\n        if row["audit_score"] < 0.08:\n            return "low_signal_review"\n        return "neutral_review"\n\n    audit["review_hint"] = audit.apply(hint, axis=1)\n\n    audit = audit.sort_values(\n        ["audit_score", "semantic_core", "nonzero_rate"],\n        ascending=False,\n    ).reset_index(drop=True)\n\n    return audit\n\n\ndef write_chat_txt(audit, group_summary, out_txt, data_path, model_dir, source_model):\n    lines = []\n\n    lines.append("========== FEATURE INVENTORY FOR REVIEW ==========")\n    lines.append(f"DATASET: {data_path}")\n    lines.append(f"MODEL_DIR: {model_dir}")\n    lines.append(f"FEATURE_SOURCE_MODEL: {source_model}")\n    lines.append(f"TOTAL_FEATURES: {len(audit)}")\n    lines.append("")\n\n    lines.append("========== GROUP SUMMARY ==========")\n    for _, r in group_summary.iterrows():\n        lines.append(\n            f"{r[\'group\']:24s} count={int(r[\'feature_count\']):4d} "\n            f"mean_score={r[\'mean_audit_score\']:.6f} "\n            f"max_score={r[\'max_audit_score\']:.6f} "\n            f"mean_nonzero={r[\'mean_nonzero_rate\']:.6f}"\n        )\n\n    lines.append("")\n    lines.append("========== TOP 120 FEATURES BY AUDIT SCORE ==========")\n\n    cols = [\n        "feature", "group", "audit_score", "abs_corr_label",\n        "mutual_info", "separation", "rf_importance",\n        "nonzero_rate", "proxy_flags", "review_hint",\n    ]\n\n    for _, r in audit.head(120)[cols].iterrows():\n        lines.append(\n            f"{r[\'audit_score\']:.6f} | {r[\'group\']:22s} | "\n            f"corr={r[\'abs_corr_label\']:.6f} mi={r[\'mutual_info\']:.6f} "\n            f"sep={r[\'separation\']:.6f} rf={r[\'rf_importance\']:.6f} "\n            f"nz={r[\'nonzero_rate\']:.4f} | "\n            f"proxy={r[\'proxy_flags\']} | {r[\'review_hint\']} | {r[\'feature\']}"\n        )\n\n    lines.append("")\n    lines.append("========== FEATURES BY GROUP ==========")\n\n    for g in group_summary["group"].tolist():\n        lines.append("")\n        lines.append(f"---- {g} ----")\n        sub = audit[audit["group"] == g].sort_values("audit_score", ascending=False)\n\n        for _, r in sub.iterrows():\n            lines.append(\n                f"{r[\'audit_score\']:.6f} | {r[\'feature\']} "\n                f"| proxy={r[\'proxy_flags\']} | hint={r[\'review_hint\']}"\n            )\n\n    Path(out_txt).write_text("\\n".join(lines), encoding="utf-8")\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--data", default="")\n    ap.add_argument("--model-dir", default="")\n    ap.add_argument("--out-dir", default="outputs/reports/feature_inventory_review")\n    args = ap.parse_args()\n\n    out_dir = Path(args.out_dir)\n    out_dir.mkdir(parents=True, exist_ok=True)\n\n    data_path = Path(args.data) if args.data else auto_find_data()\n    model_dir = Path(args.model_dir) if args.model_dir else auto_find_model_dir()\n\n    if data_path is None or not Path(data_path).exists():\n        raise SystemExit(\n            "Không tìm thấy dataset. Hãy truyền --data path/to/dataset_v6_pruned_score_corr_top260.csv"\n        )\n\n    log("[DATA]", data_path)\n    log("[MODEL_DIR]", model_dir)\n\n    feature_cols, source_model, rf_importance = infer_model_features(model_dir)\n\n    df = pd.read_csv(data_path, low_memory=False)\n\n    if feature_cols is None:\n        feature_cols = get_numeric_features(df)\n        source_model = "dataset_numeric_columns"\n\n    present = [c for c in feature_cols if c in df.columns]\n    missing = [c for c in feature_cols if c not in df.columns]\n\n    log("[INFO] dataset shape:", df.shape)\n    log("[INFO] feature cols from source:", len(feature_cols))\n    log("[INFO] present in dataset:", len(present))\n    log("[INFO] missing in dataset:", len(missing))\n\n    audit = compute_audit(df, feature_cols, rf_importance)\n\n    if audit.empty:\n        raise SystemExit("Audit rỗng.")\n\n    group_summary = (\n        audit.groupby("group", as_index=False)\n        .agg(\n            feature_count=("feature", "count"),\n            mean_audit_score=("audit_score", "mean"),\n            max_audit_score=("audit_score", "max"),\n            mean_nonzero_rate=("nonzero_rate", "mean"),\n            proxy_count=("has_proxy_flag", "sum"),\n            core_count=("semantic_core", "sum"),\n        )\n        .sort_values(["mean_audit_score", "feature_count"], ascending=False)\n    )\n\n    audit.to_csv(out_dir / "feature_inventory_all.csv", index=False)\n    audit.head(160).to_csv(out_dir / "feature_inventory_top160.csv", index=False)\n    group_summary.to_csv(out_dir / "feature_group_summary.csv", index=False)\n\n    write_chat_txt(\n        audit=audit,\n        group_summary=group_summary,\n        out_txt=out_dir / "feature_inventory_for_chat.txt",\n        data_path=data_path,\n        model_dir=model_dir,\n        source_model=source_model,\n    )\n\n    meta = {\n        "data": str(data_path),\n        "model_dir": str(model_dir),\n        "source_model": source_model,\n        "dataset_shape": list(df.shape),\n        "features_total": int(len(feature_cols)),\n        "features_present": int(len(present)),\n        "features_missing": int(len(missing)),\n        "out_dir": str(out_dir),\n    }\n\n    (out_dir / "feature_inventory_meta.json").write_text(\n        json.dumps(meta, indent=2, ensure_ascii=False),\n        encoding="utf-8",\n    )\n\n    log("")\n    log("========== GROUP SUMMARY ==========")\n    log(group_summary.to_string(index=False))\n\n    log("")\n    log("========== TOP 60 FEATURES ==========")\n    show_cols = [\n        "feature", "group", "audit_score", "abs_corr_label",\n        "mutual_info", "separation", "rf_importance",\n        "nonzero_rate", "proxy_flags", "review_hint",\n    ]\n    log(audit.head(60)[show_cols].to_string(index=False))\n\n    log("")\n    log("[OK] wrote:", out_dir / "feature_inventory_for_chat.txt")\n    log("[OK] wrote:", out_dir / "feature_inventory_all.csv")\n    log("[OK] wrote:", out_dir / "feature_group_summary.csv")\n\n\nif __name__ == "__main__":\n    main()\n', 'training/make_top260_group_drop_ablation.py': '#!/usr/bin/env python3\n# -*- coding: utf-8 -*-\n\nimport argparse\nfrom pathlib import Path\nimport re\n\nimport pandas as pd\n\n\nID_COLS = {\n    "binary", "binary_norm", "program_name", "json_file", "source_core",\n    "build_config", "function", "name_norm", "entry", "schema",\n    "merge_stage", "candidate_reasons", "label",\n}\n\n\ndef feature_group(c):\n    c = str(c)\n    cl = c.lower()\n\n    if c.startswith(("pcode_", "mnemonic_")):\n        return "opcode_like"\n\n    if c.startswith(("local_buffers_", "v6_aliases_")):\n        return "raw_buffer_alias"\n\n    if c.startswith("sinks_v6_"):\n        return "v6_sink"\n\n    if c.startswith(("v6_summary_", "v6_loop_summary_", "v6_function_role_")):\n        return "v6_summary"\n\n    if c.startswith(("sink_summary_", "buffer_summary_", "v4_summary_")):\n        return "sink_buffer_summary"\n\n    if c.startswith(("stack_", "cfg_", "summary_", "callgraph_")):\n        return "program_static"\n\n    if "risk" in cl or "safe" in cl or "guard" in cl or "overflow" in cl:\n        return "risk_safe"\n\n    if "dst" in cl or "src" in cl or "source" in cl or "size" in cl or "sink" in cl:\n        return "dataflow_sink_size"\n\n    return "other"\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--data", required=True)\n    ap.add_argument("--out-dir", default="outputs/features/top260_group_drop")\n    ap.add_argument("--min-group-size", type=int, default=8)\n    args = ap.parse_args()\n\n    data_p = Path(args.data)\n    out_dir = Path(args.out_dir)\n    out_dir.mkdir(parents=True, exist_ok=True)\n\n    df = pd.read_csv(data_p, low_memory=False)\n\n    features = [c for c in df.columns if c not in ID_COLS]\n    groups = {}\n\n    for c in features:\n        g = feature_group(c)\n        groups.setdefault(g, []).append(c)\n\n    print("========== GROUPS ==========")\n    for g, cols in sorted(groups.items(), key=lambda x: len(x[1]), reverse=True):\n        print(g, len(cols))\n\n    # full copy\n    df.to_csv(out_dir / "dataset_top260_full.csv", index=False)\n\n    for g, cols in groups.items():\n        if len(cols) < args.min_group_size:\n            continue\n\n        out = df.drop(columns=cols, errors="ignore")\n        path = out_dir / f"dataset_top260_drop_{g}.csv"\n        out.to_csv(path, index=False)\n        print("[WRITE]", path, out.shape, "dropped", len(cols))\n\n    # Một số tổ hợp thường nhiễu.\n    noisy_sets = {\n        "drop_opcode_raw_alias": ["opcode_like", "raw_buffer_alias"],\n        "drop_other": ["other"],\n        "drop_raw_alias_other": ["raw_buffer_alias", "other"],\n        "drop_opcode_other": ["opcode_like", "other"],\n    }\n\n    for name, gs in noisy_sets.items():\n        drop_cols = []\n        for g in gs:\n            drop_cols.extend(groups.get(g, []))\n\n        if not drop_cols:\n            continue\n\n        out = df.drop(columns=drop_cols, errors="ignore")\n        path = out_dir / f"dataset_top260_{name}.csv"\n        out.to_csv(path, index=False)\n        print("[WRITE]", path, out.shape, "dropped", len(drop_cols))\n\n\nif __name__ == "__main__":\n    main()\n', 'training/make_v6_noise_pruned_datasets.py': '#!/usr/bin/env python3\n# -*- coding: utf-8 -*-\n\nimport argparse\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nfrom sklearn.model_selection import StratifiedGroupKFold, train_test_split\nfrom sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier\nfrom sklearn.feature_selection import mutual_info_classif\n\n\nID_COLS = {\n    "binary", "binary_norm", "program_name", "json_file", "source_core",\n    "build_config", "function", "name_norm", "entry", "schema",\n    "merge_stage", "candidate_reasons", "label",\n}\n\n\ndef log(*x):\n    print(*x, flush=True)\n\n\ndef norm01(x):\n    x = np.asarray(x, dtype=float)\n    x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)\n    lo = float(np.min(x))\n    hi = float(np.max(x))\n    if hi <= lo:\n        return np.zeros_like(x)\n    return (x - lo) / (hi - lo)\n\n\ndef feature_group(c):\n    if c.startswith(("pcode_", "mnemonic_")):\n        return "opcode_like"\n    if c.startswith(("local_buffers_", "v6_aliases_")):\n        return "raw_buffer_alias"\n    if c.startswith(("v6_summary_", "v6_loop_summary_", "v6_function_role_")):\n        return "v6_summary"\n    if c.startswith("sinks_v6_"):\n        return "v6_sink"\n    if c.startswith(("sink_summary_", "buffer_summary_", "v4_summary_")):\n        return "sink_buffer_summary"\n    if c.startswith(("stack_", "cfg_", "summary_", "callgraph_")):\n        return "program_static"\n    if "risk" in c or "safe" in c or "guard" in c or "overflow" in c:\n        return "risk_safe"\n    return "other"\n\n\ndef group_weight(c):\n    g = feature_group(c)\n\n    if g == "opcode_like":\n        return 0.90\n    if g == "raw_buffer_alias":\n        return 0.88\n    if g in {"v6_summary", "v6_sink", "risk_safe"}:\n        return 1.08\n    if g in {"sink_buffer_summary", "program_static"}:\n        return 1.03\n\n    return 1.00\n\n\ndef split_selector(X, y, groups, seed):\n    try:\n        sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)\n        tr, va = next(sgkf.split(X, y, groups))\n        return tr, va, "stratified_group"\n    except Exception as e:\n        log("[WARN] StratifiedGroupKFold failed:", e)\n        idx = np.arange(len(y))\n        tr, va = train_test_split(\n            idx,\n            test_size=0.25,\n            random_state=seed,\n            stratify=y if len(np.unique(y)) == 2 else None,\n        )\n        return tr, va, "random_stratified"\n\n\ndef make_numeric_matrix(df):\n    feature_cols = []\n\n    for c in df.columns:\n        if c in ID_COLS:\n            continue\n\n        s = pd.to_numeric(df[c], errors="coerce")\n        if s.notna().sum() > 0:\n            feature_cols.append(c)\n\n    X = df[feature_cols].apply(pd.to_numeric, errors="coerce")\n    X = X.replace([np.inf, -np.inf], np.nan).fillna(0.0)\n\n    nunique = X.nunique(dropna=False)\n    feature_cols = [c for c in feature_cols if int(nunique[c]) > 1]\n    X = X[feature_cols].copy()\n\n    return X, feature_cols\n\n\ndef compute_scores(X, y, groups, seed, n_jobs, trees, mi_sample):\n    tr, va, split_name = split_selector(X, y, groups, seed)\n    log("[INFO] selector split:", split_name)\n    log("[INFO] selector train:", len(tr), "hold:", len(va))\n\n    Xtr = X.iloc[tr]\n    ytr = y[tr]\n\n    log("[1] Train selector RandomForest")\n    rf = RandomForestClassifier(\n        n_estimators=trees,\n        max_features="sqrt",\n        min_samples_leaf=1,\n        class_weight="balanced_subsample",\n        n_jobs=n_jobs,\n        random_state=seed,\n    )\n    rf.fit(Xtr, ytr)\n\n    log("[2] Train selector ExtraTrees")\n    et = ExtraTreesClassifier(\n        n_estimators=trees,\n        max_features="sqrt",\n        min_samples_leaf=1,\n        class_weight="balanced",\n        n_jobs=n_jobs,\n        random_state=seed + 17,\n    )\n    et.fit(Xtr, ytr)\n\n    log("[3] Mutual information")\n    rng = np.random.default_rng(seed)\n\n    if len(tr) > mi_sample:\n        mi_idx = rng.choice(tr, size=mi_sample, replace=False)\n    else:\n        mi_idx = tr\n\n    mi = mutual_info_classif(\n        X.iloc[mi_idx],\n        y[mi_idx],\n        random_state=seed,\n        discrete_features=False,\n    )\n\n    log("[4] Class separation")\n    pos = Xtr[ytr == 1]\n    neg = Xtr[ytr == 0]\n\n    mean_diff = (pos.mean(axis=0) - neg.mean(axis=0)).abs()\n    std_all = Xtr.std(axis=0).replace(0, np.nan)\n    sep = (mean_diff / std_all).replace([np.inf, -np.inf], np.nan).fillna(0.0).values\n\n    nonzero_ratio = (X != 0).mean(axis=0).values\n    unique_count = X.nunique(dropna=False).values\n\n    rf_n = norm01(rf.feature_importances_)\n    et_n = norm01(et.feature_importances_)\n    mi_n = norm01(mi)\n    sep_n = norm01(sep)\n\n    base_score = 0.42 * rf_n + 0.32 * et_n + 0.16 * mi_n + 0.10 * sep_n\n\n    features = list(X.columns)\n    weights = np.array([group_weight(c) for c in features], dtype=float)\n    final_score = base_score * weights\n\n    rank = pd.DataFrame({\n        "feature": features,\n        "group": [feature_group(c) for c in features],\n        "score": final_score,\n        "base_score": base_score,\n        "rf_importance": rf.feature_importances_,\n        "et_importance": et.feature_importances_,\n        "mutual_info": mi,\n        "class_separation": sep,\n        "nonzero_ratio": nonzero_ratio,\n        "unique_count": unique_count,\n        "group_weight": weights,\n    })\n\n    rank = rank.sort_values("score", ascending=False).reset_index(drop=True)\n    return rank, tr\n\n\ndef corr_prune_order(X_train, ranked_features, score_map, threshold):\n    cols = [c for c in ranked_features if c in X_train.columns]\n\n    log("[INFO] correlation matrix cols:", len(cols))\n    corr = X_train[cols].corr().abs().fillna(0.0)\n\n    selected = []\n\n    for c in cols:\n        keep = True\n\n        for s in selected:\n            if float(corr.loc[c, s]) >= threshold:\n                keep = False\n                break\n\n        if keep:\n            selected.append(c)\n\n    return selected\n\n\ndef write_dataset(df, out_dir, name, features):\n    meta = [c for c in df.columns if c in ID_COLS]\n    cols = []\n\n    for c in meta + list(features):\n        if c in df.columns and c not in cols:\n            cols.append(c)\n\n    out = out_dir / f"dataset_v6_pruned_{name}.csv"\n    df[cols].to_csv(out, index=False)\n    log("[WRITE]", out, df[cols].shape)\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n\n    ap.add_argument("--data", required=True)\n    ap.add_argument("--out-dir", default="outputs/features/v6_pruned")\n    ap.add_argument("--group-col", default="source_core")\n    ap.add_argument("--seed", type=int, default=1337)\n    ap.add_argument("--n-jobs", type=int, default=4)\n    ap.add_argument("--trees", type=int, default=500)\n    ap.add_argument("--mi-sample", type=int, default=20000)\n    ap.add_argument("--corr-threshold", type=float, default=0.995)\n    ap.add_argument("--topk", default="220,260,300,340")\n\n    args = ap.parse_args()\n\n    data = Path(args.data)\n    out_dir = Path(args.out_dir)\n    out_dir.mkdir(parents=True, exist_ok=True)\n\n    log("[LOAD]", data)\n    df = pd.read_csv(data, low_memory=False)\n\n    df["label"] = pd.to_numeric(df["label"], errors="coerce")\n    df = df[df["label"].isin([0, 1])].copy()\n    df["label"] = df["label"].astype(int)\n\n    y = df["label"].astype(int).values\n\n    if args.group_col in df.columns:\n        groups = df[args.group_col].astype(str).values\n    else:\n        groups = np.arange(len(df)).astype(str)\n\n    log("[INFO] dataset:", df.shape)\n    log("[INFO] labels:")\n    log(df["label"].value_counts().to_string())\n\n    X, features = make_numeric_matrix(df)\n\n    log("[INFO] usable nonconstant numeric:", len(features))\n\n    rank, tr = compute_scores(\n        X=X,\n        y=y,\n        groups=groups,\n        seed=args.seed,\n        n_jobs=args.n_jobs,\n        trees=args.trees,\n        mi_sample=args.mi_sample,\n    )\n\n    rank_path = out_dir / "v6_feature_noise_ranking.csv"\n    rank.to_csv(rank_path, index=False)\n    log("[OK] ranking:", rank_path)\n\n    score_map = dict(zip(rank["feature"], rank["score"]))\n\n    ranked_features = rank["feature"].tolist()\n    Xtr = X.iloc[tr]\n\n    corr_order = corr_prune_order(\n        X_train=Xtr,\n        ranked_features=ranked_features,\n        score_map=score_map,\n        threshold=args.corr_threshold,\n    )\n\n    corr_rank = pd.DataFrame({\n        "feature": corr_order,\n        "corr_pruned_order": np.arange(1, len(corr_order) + 1),\n        "score": [score_map[c] for c in corr_order],\n        "group": [feature_group(c) for c in corr_order],\n    })\n\n    corr_rank_path = out_dir / "v6_feature_corr_pruned_order.csv"\n    corr_rank.to_csv(corr_rank_path, index=False)\n    log("[OK] corr order:", corr_rank_path)\n    log("[INFO] corr-pruned features:", len(corr_order))\n\n    topks = [int(x.strip()) for x in args.topk.split(",") if x.strip()]\n\n    drop_group_clean = [\n        c for c in df.columns\n        if c.startswith(("pcode_", "mnemonic_", "local_buffers_", "v6_aliases_"))\n    ]\n\n    group_clean_features = [\n        c for c in features\n        if c not in set(drop_group_clean)\n    ]\n\n    write_dataset(df, out_dir, "group_clean", group_clean_features)\n\n    drop_semantic = [\n        c for c in df.columns\n        if c.startswith(("pcode_", "mnemonic_", "local_buffers_", "v6_aliases_"))\n    ]\n\n    semantic_features = [\n        c for c in features\n        if c not in set(drop_semantic)\n    ]\n\n    write_dataset(df, out_dir, "semantic_clean", semantic_features)\n\n    for k in topks:\n        feats = ranked_features[:min(k, len(ranked_features))]\n        write_dataset(df, out_dir, f"score_top{k}", feats)\n\n    for k in topks:\n        feats = corr_order[:min(k, len(corr_order))]\n        write_dataset(df, out_dir, f"score_corr_top{k}", feats)\n\n    security_groups = {\n        "v6_summary",\n        "v6_sink",\n        "risk_safe",\n        "sink_buffer_summary",\n        "program_static",\n    }\n\n    sec_ranked = [\n        c for c in corr_order\n        if feature_group(c) in security_groups\n    ]\n\n    for k in [220, 260, 300]:\n        feats = sec_ranked[:min(k, len(sec_ranked))]\n        write_dataset(df, out_dir, f"security_corr_top{k}", feats)\n\n    log("[DONE] wrote datasets to:", out_dir)\n\n\nif __name__ == "__main__":\n    main()\n', 'training/protocol.py': '"""Explicit group partitions shared by feature selection and training."""\nimport hashlib\nimport json\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nfrom sklearn.model_selection import StratifiedGroupKFold\n\nPARTITIONS = (\'train\', \'meta\', \'select\', \'test\')\n\n\ndef sha256(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for chunk in iter(lambda: stream.read(1024 * 1024), b\'\'):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef validate_partitions(df, manifest):\n    group_col = manifest[\'group_col\']\n    if group_col not in df or df[group_col].isna().any():\n        raise ValueError(\'Missing group identifiers.\')\n    if \'label\' not in df or not df[\'label\'].isin([0, 1]).all():\n        raise ValueError(\'Every row must have a binary 0/1 label.\')\n    ids, groups = set(), set()\n    for name in PARTITIONS:\n        rows = manifest[\'partitions\'][name]\n        if not rows or any(type(i) is not int or not 0 <= i < len(df) for i in rows):\n            raise ValueError(f\'Invalid row indexes in {name}.\')\n        if len(set(rows)) != len(rows) or ids.intersection(rows):\n            raise ValueError(\'Partitions overlap by row.\')\n        group_set = set(df.iloc[rows][group_col].astype(str))\n        if groups.intersection(group_set):\n            raise ValueError(\'Partitions overlap by group.\')\n        if set(df.iloc[rows][\'label\']) != {0, 1}:\n            raise ValueError(f\'{name} does not contain both classes; use more groups.\')\n        ids.update(rows)\n        groups.update(group_set)\n    if ids != set(range(len(df))):\n        raise ValueError(\'Partitions must cover every input row exactly once.\')\n\n\ndef make_partitions(df, seed=42, group_col=\'source_core\'):\n    if group_col not in df or df[group_col].isna().any():\n        raise ValueError(f\'Input requires nonempty {group_col}.\')\n    if not df[\'label\'].isin([0, 1]).all():\n        raise ValueError(\'Labels must be 0/1, with no missing values.\')\n    y = df[\'label\'].to_numpy()\n    groups = df[group_col].astype(str).to_numpy()\n    idx = np.arange(len(df))\n    def split(rows, folds, random_state):\n        if len(set(groups[rows])) < folds:\n            raise ValueError(\'Too few groups for the requested split; no random-row fallback.\')\n        splitter = StratifiedGroupKFold(folds, shuffle=True, random_state=random_state)\n        a, b = next(splitter.split(rows, y[rows], groups[rows]))\n        return rows[a], rows[b]\n    train_val, test = split(idx, 5, seed)\n    train, val = split(train_val, 4, seed + 1)\n    meta, select = split(val, 2, seed + 77)\n    result = {\'protocol\': \'group_split_before_feature_selection_v1\',\n              \'random_state\': seed, \'group_col\': group_col, \'row_count\': len(df),\n              \'partitions\': {k: v.tolist() for k, v in\n                             zip(PARTITIONS, (train, meta, select, test))}}\n    validate_partitions(df, result)\n    return result\n\n\ndef load_partitions(path, data_path, df):\n    manifest = json.loads(Path(path).read_text(encoding=\'utf-8\'))\n    if manifest[\'dataset_sha256\'] != sha256(data_path):\n        raise ValueError(\'Dataset hash does not match the saved split manifest.\')\n    validate_partitions(df, manifest)\n    return {name: np.asarray(manifest[\'partitions\'][name], dtype=int) for name in PARTITIONS}\n', 'training/run_experiment.py': '"""Split first, select features only on train, then fit the supplied ensemble suite."""\nimport argparse\nimport json\nfrom pathlib import Path\nimport subprocess\nimport sys\nimport pandas as pd\n\ntry:\n    from .protocol import make_partitions, sha256\nexcept ImportError:\n    from protocol import make_partitions, sha256\n\nHERE = Path(__file__).resolve().parent\nID_COLUMNS = {\'binary\', \'binary_norm\', \'program_name\', \'json_file\', \'source_core\',\n              \'build_config\', \'function\', \'name_norm\', \'entry\', \'schema\',\n              \'merge_stage\', \'candidate_reasons\', \'label\'}\n\n\ndef run(script, *args):\n    subprocess.run([sys.executable, str(HERE / script), *map(str, args)], check=True)\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--data\', required=True, help=\'Original full feature CSV; not the 200-row demo.\')\n    parser.add_argument(\'--out-dir\', required=True, help=\'New directory for one experiment.\')\n    parser.add_argument(\'--features\', type=int, choices=[160, 260], default=260)\n    parser.add_argument(\'--seed\', type=int, default=42)\n    parser.add_argument(\'--n-jobs\', type=int, default=4)\n    parser.add_argument(\'--selector-trees\', type=int, default=500)\n    parser.add_argument(\'--rf-trees\', type=int, default=300)\n    parser.add_argument(\'--xgb-trees\', type=int, default=450)\n    parser.add_argument(\'--xgb-device\', choices=[\'cpu\', \'cuda\'], default=\'cpu\')\n    parser.add_argument(\'--weight-step\', type=float, default=.05)\n    parser.add_argument(\'--prepare-only\', action=\'store_true\', help=\'Validate and save outer partitions only.\')\n    args = parser.parse_args()\n    data = Path(args.data).resolve()\n    out = Path(args.out_dir).resolve()\n    if out.exists() and any(out.iterdir()):\n        raise ValueError(\'Output directory must be new or empty; choose a new run name.\')\n    out.mkdir(parents=True, exist_ok=True)\n    df = pd.read_csv(data, low_memory=False)\n    manifest = make_partitions(df, args.seed)\n    manifest[\'source_dataset_sha256\'] = sha256(data)\n    manifest[\'dataset_sha256\'] = manifest[\'source_dataset_sha256\']\n    (out / \'source_split_manifest.json\').write_text(json.dumps(manifest, indent=2), encoding=\'utf-8\')\n    if args.prepare_only:\n        print(\'Group partitions verified; no feature selection or training performed.\')\n        return\n\n    # Only this partition is visible to all feature-ranking/audit scripts.\n    train_csv = out / \'train_for_selection.csv\'\n    train_df = df.iloc[manifest[\'partitions\'][\'train\']]\n    train_df.to_csv(train_csv, index=False)\n    selected_dir = out / \'feature_selection\'\n    run(\'make_v6_noise_pruned_datasets.py\', \'--data\', train_csv, \'--out-dir\', selected_dir,\n        \'--group-col\', \'source_core\', \'--seed\', args.seed, \'--n-jobs\', args.n_jobs,\n        \'--trees\', args.selector_trees, \'--corr-threshold\', .995, \'--topk\', 260)\n    selected_csv = selected_dir / \'dataset_v6_pruned_score_corr_top260.csv\'\n    if args.features == 160:\n        audit_dir = out / \'train_feature_audit\'\n        run(\'export_feature_inventory_for_review.py\', \'--data\', selected_csv,\n            \'--model-dir\', out / \'no_pretrained_models\', \'--out-dir\', audit_dir)\n        final_dir = out / \'feature_selection_160\'\n        run(\'select_final_security_features.py\', \'--data\', selected_csv,\n            \'--audit\', audit_dir / \'feature_inventory_all.csv\', \'--out-dir\', final_dir,\n            \'--target-k\', 160, \'--corr-threshold\', .965)\n        candidates = list(final_dir.glob(\'dataset_final_security_*.csv\'))\n        if len(candidates) != 1:\n            raise ValueError(\'Cannot identify the selected 160-feature dataset.\')\n        selected_csv = candidates[0]\n    columns = pd.read_csv(selected_csv, nrows=0).columns.tolist()\n    feature_cols = [c for c in columns if c not in ID_COLUMNS]\n    if len(feature_cols) != args.features:\n        raise ValueError(f\'Only {len(feature_cols)} features available; requested {args.features}.\')\n    full_csv = out / \'dataset_selected.csv\'\n    df.loc[:, columns].to_csv(full_csv, index=False)\n    manifest[\'dataset_sha256\'] = sha256(full_csv)\n    manifest[\'selected_features\'] = feature_cols\n    split_path = out / \'split_manifest.json\'\n    split_path.write_text(json.dumps(manifest, indent=2), encoding=\'utf-8\')\n    run(\'train_sbo_v5_ensemble_suite_kaggle.py\', \'--data\', full_csv, \'--out-dir\', out / \'models\',\n        \'--split-manifest\', split_path, \'--n-jobs\', args.n_jobs, \'--random-state\', args.seed,\n        \'--rf-trees\', args.rf_trees, \'--xgb-trees\', args.xgb_trees,\n        \'--xgb-device\', args.xgb_device, \'--weight-step\', args.weight_step, \'--recall-min\', .95)\n\n\nif __name__ == \'__main__\':\n    main()\n', 'training/select_final_security_features.py': '#!/usr/bin/env python3\n# -*- coding: utf-8 -*-\n\nimport argparse\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\n\nID_COLS = [\n    "binary", "binary_norm", "program_name", "json_file", "source_core",\n    "build_config", "function", "name_norm", "entry", "schema",\n    "merge_stage", "candidate_reasons", "label",\n]\n\n\nGROUP_RATIO = {\n    "v6_sink": 0.27,\n    "risk_safe": 0.20,\n    "sink_buffer_summary": 0.12,\n    "v6_summary": 0.12,\n    "raw_buffer_alias": 0.10,\n    "dataflow_sink_size": 0.08,\n    "opcode_like": 0.06,\n    "stack_memory": 0.04,\n    "program_static": 0.01,\n    "callgraph_context": 0.00,\n    "cfg_control_flow": 0.00,\n}\n\n\nSEMANTIC_BONUS = {\n    "v6_sink": 0.08,\n    "risk_safe": 0.07,\n    "sink_buffer_summary": 0.07,\n    "v6_summary": 0.06,\n    "raw_buffer_alias": 0.06,\n    "stack_memory": 0.06,\n    "dataflow_sink_size": 0.02,\n    "opcode_like": 0.00,\n    "program_static": -0.04,\n    "callgraph_context": -0.06,\n    "cfg_control_flow": -0.10,\n}\n\n\ndef log(*x):\n    print(*x, flush=True)\n\n\ndef proxy_penalty(proxy_flags):\n    s = str(proxy_flags or "")\n    if not s:\n        return 0.0\n\n    penalty = 0.0\n\n    if "line_index" in s:\n        penalty += 0.10\n    if "stack_offset" in s:\n        penalty += 0.08\n    if "size_proxy" in s:\n        penalty += 0.05\n    if "generic_count" in s:\n        penalty += 0.04\n\n    return penalty\n\n\ndef build_quota(target_k):\n    raw = {g: int(round(target_k * r)) for g, r in GROUP_RATIO.items()}\n\n    # Điều chỉnh cho tổng đúng target_k.\n    diff = target_k - sum(raw.values())\n\n    priority = [\n        "v6_sink", "risk_safe", "sink_buffer_summary", "v6_summary",\n        "raw_buffer_alias", "dataflow_sink_size", "opcode_like", "stack_memory"\n    ]\n\n    i = 0\n    while diff != 0:\n        g = priority[i % len(priority)]\n        if diff > 0:\n            raw[g] += 1\n            diff -= 1\n        else:\n            if raw[g] > 0:\n                raw[g] -= 1\n                diff += 1\n        i += 1\n\n    return raw\n\n\ndef corr_with_selected(X, candidate, selected, corr_cache):\n    if not selected:\n        return 0.0\n\n    max_corr = 0.0\n\n    x = X[candidate].values\n\n    sx = np.std(x)\n    if sx < 1e-12:\n        return 1.0\n\n    for s in selected:\n        key = tuple(sorted((candidate, s)))\n\n        if key in corr_cache:\n            c = corr_cache[key]\n        else:\n            y = X[s].values\n            sy = np.std(y)\n\n            if sy < 1e-12:\n                c = 1.0\n            else:\n                c = abs(np.corrcoef(x, y)[0, 1])\n                if not np.isfinite(c):\n                    c = 0.0\n\n            corr_cache[key] = c\n\n        max_corr = max(max_corr, c)\n\n        if max_corr >= 0.985:\n            break\n\n    return max_corr\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--data", required=True)\n    ap.add_argument("--audit", required=True)\n    ap.add_argument("--out-dir", default="outputs/features/final_selected")\n    ap.add_argument("--target-k", type=int, default=160)\n    ap.add_argument("--corr-threshold", type=float, default=0.965)\n    ap.add_argument("--sample-corr", type=int, default=12000)\n    args = ap.parse_args()\n\n    out_dir = Path(args.out_dir)\n    out_dir.mkdir(parents=True, exist_ok=True)\n\n    df = pd.read_csv(args.data, low_memory=False)\n    audit = pd.read_csv(args.audit)\n    audit["proxy_flags"] = audit["proxy_flags"].fillna("")\n\n    audit = audit[audit["feature"].isin(df.columns)].copy()\n\n    if audit.empty:\n        raise SystemExit("Audit rỗng hoặc feature không khớp dataset.")\n\n    audit["proxy_penalty"] = audit["proxy_flags"].apply(proxy_penalty)\n    audit["semantic_bonus"] = audit["group"].map(SEMANTIC_BONUS).fillna(0.0)\n\n    audit["final_select_score"] = (\n        audit["audit_score"].astype(float)\n        + audit["semantic_bonus"]\n        - audit["proxy_penalty"]\n    )\n\n    audit = audit.sort_values(\n        ["final_select_score", "audit_score", "nonzero_rate"],\n        ascending=False,\n    ).reset_index(drop=True)\n\n    feature_cols = audit["feature"].tolist()\n\n    # Dùng sample để tính correlation cho nhanh.\n    if len(df) > args.sample_corr:\n        if "label" in df.columns:\n            sample_df = (\n                df.groupby("label", group_keys=False)\n                .apply(lambda x: x.sample(\n                    min(len(x), max(1000, args.sample_corr // 2)),\n                    random_state=1337\n                ))\n            )\n        else:\n            sample_df = df.sample(args.sample_corr, random_state=1337)\n    else:\n        sample_df = df\n\n    Xcorr = sample_df[feature_cols].apply(pd.to_numeric, errors="coerce")\n    Xcorr = Xcorr.replace([np.inf, -np.inf], np.nan).fillna(0.0)\n\n    quota = build_quota(args.target_k)\n\n    selected = []\n    corr_cache = {}\n\n    log("========== QUOTA ==========")\n    for g, q in quota.items():\n        log(f"{g:24s} {q}")\n\n    log("")\n    log("========== SELECTING ==========")\n\n    # Chọn theo quota từng nhóm.\n    for group, q in quota.items():\n        if q <= 0:\n            continue\n\n        sub = audit[audit["group"] == group].copy()\n\n        n_before = len(selected)\n        for _, row in sub.iterrows():\n            if len(selected) - n_before >= q:\n                break\n\n            f = row["feature"]\n\n            if f in selected:\n                continue\n\n            # Correlation pruning toàn cục.\n            max_corr = corr_with_selected(Xcorr, f, selected, corr_cache)\n\n            if max_corr >= args.corr_threshold:\n                continue\n\n            selected.append(f)\n\n        log(f"{group:24s} selected={len(selected) - n_before:3d}/{q:3d}")\n\n    # Nếu chưa đủ target_k thì fill từ feature còn lại, vẫn tránh trùng lặp quá cao.\n    for _, row in audit.iterrows():\n        if len(selected) >= args.target_k:\n            break\n\n        f = row["feature"]\n\n        if f in selected:\n            continue\n\n        max_corr = corr_with_selected(Xcorr, f, selected, corr_cache)\n\n        if max_corr >= 0.985:\n            continue\n\n        selected.append(f)\n\n    selected = selected[:args.target_k]\n\n    keep_cols = [c for c in ID_COLS if c in df.columns] + selected\n    out_df = df[keep_cols].copy()\n\n    out_csv = out_dir / f"dataset_final_security_{len(selected)}.csv"\n    out_list = out_dir / f"feature_list_final_security_{len(selected)}.txt"\n    out_audit = out_dir / f"selected_feature_audit_{len(selected)}.csv"\n    out_summary = out_dir / f"selected_group_summary_{len(selected)}.csv"\n\n    out_df.to_csv(out_csv, index=False)\n    Path(out_list).write_text("\\n".join(selected), encoding="utf-8")\n\n    selected_audit = audit[audit["feature"].isin(selected)].copy()\n    selected_audit = selected_audit.sort_values(\n        ["group", "final_select_score"],\n        ascending=[True, False]\n    )\n\n    selected_audit.to_csv(out_audit, index=False)\n\n    summary = (\n        selected_audit.groupby("group", as_index=False)\n        .agg(\n            feature_count=("feature", "count"),\n            mean_audit_score=("audit_score", "mean"),\n            max_audit_score=("audit_score", "max"),\n            mean_final_select_score=("final_select_score", "mean"),\n            proxy_count=("proxy_flags", lambda x: sum(str(v) != "" for v in x)),\n        )\n        .sort_values(["feature_count", "mean_audit_score"], ascending=False)\n    )\n\n    summary.to_csv(out_summary, index=False)\n\n    log("")\n    log("========== SELECTED GROUP SUMMARY ==========")\n    log(summary.to_string(index=False))\n\n    log("")\n    log("[OK] dataset:", out_csv)\n    log("[OK] feature list:", out_list)\n    log("[OK] selected audit:", out_audit)\n    log("[OK] summary:", out_summary)\n\n\nif __name__ == "__main__":\n    main()\n', 'training/train_sbo_v5_ensemble_suite_kaggle.py': '#!/usr/bin/env python3\n# -*- coding: utf-8 -*-\n\nimport argparse\nimport json\nimport warnings\nimport os\nimport sys\nimport importlib.metadata\nfrom pathlib import Path\nfrom collections import OrderedDict\n\nimport joblib\nimport numpy as np\nimport pandas as pd\n\nfrom sklearn.model_selection import train_test_split, StratifiedGroupKFold\nfrom sklearn.metrics import (\n    accuracy_score, balanced_accuracy_score, precision_score, recall_score,\n    f1_score, roc_auc_score, average_precision_score, confusion_matrix\n)\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.tree import DecisionTreeClassifier\nfrom sklearn.ensemble import RandomForestClassifier\n\n# Keep compatibility and convergence warnings visible.\n\ntry:\n    from xgboost import XGBClassifier\n    HAS_XGB = True\nexcept Exception:\n    HAS_XGB = False\n\n\nID_COLS = {\n    "binary", "binary_norm", "program_name", "json_file", "source_core",\n    "build_config", "function", "name_norm", "entry", "schema",\n    "merge_stage", "candidate_reasons", "label",\n}\n\nRISK_TOKENS = [\n    "vuln_signal", "safe_signal", "risk_score", "safe_score", "risk_minus_safe",\n    "rule_", "strict_", "txt_", "interproc", "ip_", "loop", "alias", "alloca",\n    "high_quality_guard", "literal_stack", "bounded_stack", "external_source",\n    "size_overflow", "unbounded_stack",\n]\n\n\ndef log(*xs):\n    print(*xs, flush=True)\n\n\ndef parse_float_list(s):\n    return [float(x.strip()) for x in str(s).split(",") if x.strip()]\n\n\ndef safe_auc(y, p):\n    try:\n        return roc_auc_score(y, p)\n    except Exception:\n        return np.nan\n\n\ndef safe_ap(y, p):\n    try:\n        return average_precision_score(y, p)\n    except Exception:\n        return np.nan\n\n\ndef eval_prob(y, p, thr):\n    yp = (p >= thr).astype(int)\n    tn, fp, fn, tp = confusion_matrix(y, yp, labels=[0, 1]).ravel()\n\n    return {\n        "test_accuracy": accuracy_score(y, yp),\n        "test_balanced_accuracy": balanced_accuracy_score(y, yp),\n        "test_precision_vulnerable": precision_score(y, yp, zero_division=0),\n        "test_recall_vulnerable": recall_score(y, yp, zero_division=0),\n        "test_f1_vulnerable": f1_score(y, yp, zero_division=0),\n        "test_macro_f1": f1_score(y, yp, average="macro", zero_division=0),\n        "test_roc_auc": safe_auc(y, p),\n        "test_average_precision": safe_ap(y, p),\n        "test_tn": int(tn),\n        "test_fp": int(fp),\n        "test_fn": int(fn),\n        "test_tp": int(tp),\n    }\n\n\ndef val_metric(y, p, thr):\n    yp = (p >= thr).astype(int)\n    tn, fp, fn, tp = confusion_matrix(y, yp, labels=[0, 1]).ravel()\n\n    return {\n        "threshold": float(thr),\n        "f1": f1_score(y, yp, zero_division=0),\n        "recall": recall_score(y, yp, zero_division=0),\n        "precision": precision_score(y, yp, zero_division=0),\n        "accuracy": accuracy_score(y, yp),\n        "tn": int(tn),\n        "fp": int(fp),\n        "fn": int(fn),\n        "tp": int(tp),\n    }\n\n\ndef better(cand, best, recall_min=None):\n    if best is None:\n        return True\n\n    if recall_min is not None:\n        cand_ok = cand["recall"] >= recall_min\n        best_ok = best["recall"] >= recall_min\n\n        if cand_ok and not best_ok:\n            return True\n        if best_ok and not cand_ok:\n            return False\n\n    if cand["f1"] > best["f1"]:\n        return True\n\n    if cand["f1"] == best["f1"]:\n        if cand["recall"] > best["recall"]:\n            return True\n\n        if cand["recall"] == best["recall"]:\n            if cand["fp"] < best["fp"]:\n                return True\n\n            if cand["fp"] == best["fp"] and cand["precision"] > best["precision"]:\n                return True\n\n    return False\n\n\ndef tune_threshold(y, p, recall_min=None):\n    best = None\n\n    for thr in np.arange(0.05, 0.951, 0.005):\n        cand = val_metric(y, p, thr)\n\n        if better(cand, best, recall_min):\n            best = cand\n\n    return best\n\n\ndef group_col(df):\n    for c in ["source_core", "binary_norm", "binary", "program_name", "json_file"]:\n        if c in df.columns:\n            return c\n\n    return None\n\n\ndef split_data(y, groups, random_state):\n    idx = np.arange(len(y))\n\n    if groups is None or len(set(groups)) < 5:\n        train_val, test = train_test_split(\n            idx,\n            test_size=0.20,\n            stratify=y,\n            random_state=random_state,\n        )\n\n        train, val = train_test_split(\n            train_val,\n            test_size=0.25,\n            stratify=y[train_val],\n            random_state=random_state,\n        )\n\n        return train, val, test, "stratified_random"\n\n    sgkf = StratifiedGroupKFold(\n        n_splits=5,\n        shuffle=True,\n        random_state=random_state,\n    )\n\n    train_val, test = next(sgkf.split(idx, y, groups))\n\n    sgkf2 = StratifiedGroupKFold(\n        n_splits=4,\n        shuffle=True,\n        random_state=random_state + 1,\n    )\n\n    local_train, local_val = next(\n        sgkf2.split(\n            np.arange(len(train_val)),\n            y[train_val],\n            groups[train_val],\n        )\n    )\n\n    return train_val[local_train], train_val[local_val], test, "stratified_group_by_binary_core"\n\n\ndef load_dataset(path):\n    log("[1] Load dataset...")\n\n    df = pd.read_csv(path, low_memory=False)\n\n    if "label" not in df.columns:\n        raise SystemExit("[ERROR] Không thấy cột label")\n\n    df["label"] = pd.to_numeric(df["label"], errors="coerce")\n    df = df.dropna(subset=["label"]).copy()\n    df["label"] = df["label"].astype(int)\n\n    gcol = group_col(df)\n    groups = df[gcol].astype(str).values if gcol else None\n\n    y = df["label"].values\n\n    log("[INFO] dataset shape:", df.shape)\n    log("[INFO] label counts:")\n    log(df["label"].value_counts().to_string())\n\n    x_data = {}\n\n    for c in df.columns:\n        if c in ID_COLS:\n            continue\n\n        s = pd.to_numeric(df[c], errors="coerce")\n\n        if s.notna().sum() == 0:\n            continue\n\n        x_data[c] = s.astype(np.float32)\n\n    X = pd.DataFrame(x_data)\n    X = X.replace([np.inf, -np.inf], np.nan).fillna(0.0)\n\n    nunique = X.nunique(dropna=False)\n    const_cols = nunique[nunique <= 1].index.tolist()\n\n    if const_cols:\n        X = X.drop(columns=const_cols)\n\n    log("[2] Prepare X/y...")\n    log("[INFO] group col:", gcol)\n    log("[INFO] usable numeric features:", X.shape[1])\n    log("[INFO] dropped constant features:", len(const_cols))\n\n    return df, X, y, groups, gcol, const_cols\n\n\ndef build_models(scale_pos_weight, n_jobs, random_state, rf_trees=500, xgb_trees=650, xgb_device="cpu"):\n    if not HAS_XGB:\n        raise SystemExit("[ERROR] Chưa cài xgboost. Trên Kaggle thường có sẵn; nếu thiếu hãy bật Internet rồi chạy: pip install xgboost")\n\n    # Kaggle: GPU chỉ giúp XGBoost, còn RandomForest/DecisionTree vẫn chạy CPU.\n    # Để ổn định trên mọi version XGBoost, mặc định dùng CPU hist.\n    xgb_params = dict(\n        n_estimators=int(xgb_trees),\n        max_depth=5,\n        learning_rate=0.035,\n        subsample=0.90,\n        colsample_bytree=0.90,\n        reg_lambda=2.0,\n        reg_alpha=0.05,\n        objective="binary:logistic",\n        eval_metric="logloss",\n        tree_method="hist",\n        random_state=random_state,\n        n_jobs=n_jobs,\n        scale_pos_weight=scale_pos_weight,\n    )\n\n    if str(xgb_device).lower() in {"cuda", "gpu"}:\n        # XGBoost >= 2.x: device="cuda"; XGBoost cũ: tree_method="gpu_hist".\n        # Nếu Kaggle báo lỗi, chạy lại với --xgb-device cpu.\n        xgb_params["device"] = "cuda"\n\n    return OrderedDict({\n        "LogisticRegression": Pipeline([\n            ("scaler", StandardScaler()),\n            ("clf", LogisticRegression(\n                max_iter=3000,\n                class_weight="balanced",\n                solver="lbfgs",\n            )),\n        ]),\n\n        "DecisionTree": DecisionTreeClassifier(\n            random_state=random_state,\n            class_weight="balanced",\n            min_samples_leaf=2,\n        ),\n\n        "RandomForest": RandomForestClassifier(\n            n_estimators=int(rf_trees),\n            random_state=random_state,\n            class_weight="balanced_subsample",\n            n_jobs=n_jobs,\n            max_features="sqrt",\n            min_samples_leaf=1,\n        ),\n\n        "XGBoost": XGBClassifier(**xgb_params),\n    })\n\n\ndef prob(model, X):\n    return model.predict_proba(X)[:, 1]\n\n\ndef signed_log1p(x):\n    x = np.asarray(x, dtype=float)\n    return np.sign(x) * np.log1p(np.abs(x))\n\n\ndef select_risk_cols(X):\n    cols = []\n\n    for c in X.columns:\n        if any(t in c for t in RISK_TOKENS):\n            cols.append(c)\n\n    return cols[:160]\n\n\ndef meta_matrix(Xdf, probs, risk_cols=None):\n    out = pd.DataFrame(index=Xdf.index)\n\n    names = list(probs.keys())\n\n    for n in names:\n        p = np.asarray(probs[n], dtype=float)\n        out[f"p_{n}"] = p\n        out[f"u_{n}"] = 1.0 - 2.0 * np.abs(p - 0.5)\n\n    for i in range(len(names)):\n        for j in range(i + 1, len(names)):\n            a = names[i]\n            b = names[j]\n            out[f"diff_{a}_minus_{b}"] = np.asarray(probs[a]) - np.asarray(probs[b])\n            out[f"absdiff_{a}_{b}"] = np.abs(np.asarray(probs[a]) - np.asarray(probs[b]))\n\n    if risk_cols:\n        for c in risk_cols:\n            if c in Xdf.columns:\n                out[c] = signed_log1p(Xdf[c].values)\n\n    return out.replace([np.inf, -np.inf], np.nan).fillna(0.0)\n\n\ndef weight_grid(k, step):\n    units = int(round(1.0 / step))\n    cur = []\n\n    def rec(pos, remain):\n        if pos == k - 1:\n            yield cur + [remain]\n            return\n\n        for v in range(remain + 1):\n            cur.append(v)\n            yield from rec(pos + 1, remain - v)\n            cur.pop()\n\n    for ints in rec(0, units):\n        yield np.array(ints, dtype=float) / units\n\n\ndef optimize_weighted(y, probs, names, recall_min, step, max_lr_weight=0.10):\n    P = np.vstack([probs[n] for n in names]).T\n    best = None\n\n    for w in weight_grid(len(names), step):\n        if "LogisticRegression" in names:\n            idx = names.index("LogisticRegression")\n            if w[idx] > max_lr_weight:\n                continue\n\n        p = P.dot(w)\n        cand = tune_threshold(y, p, recall_min=recall_min)\n        cand["weights"] = {names[i]: float(w[i]) for i in range(len(names))}\n\n        if better(cand, best, recall_min):\n            best = cand\n\n    return best\n\n\ndef apply_weighted(probs, info):\n    p = None\n\n    for name, w in info["weights"].items():\n        if p is None:\n            p = w * np.asarray(probs[name])\n        else:\n            p += w * np.asarray(probs[name])\n\n    return p\n\n\ndef train_stack(X_meta, y_meta, X_select, y_select, X_test, recall_min, C):\n    meta = Pipeline([\n        ("scaler", StandardScaler()),\n        ("clf", LogisticRegression(\n            max_iter=3000,\n            C=C,\n            class_weight="balanced",\n        )),\n    ])\n\n    meta.fit(X_meta, y_meta)\n\n    p_select = meta.predict_proba(X_select)[:, 1]\n    thr = tune_threshold(y_select, p_select, recall_min=recall_min)\n\n    p_test = meta.predict_proba(X_test)[:, 1]\n\n    return meta, thr, p_test\n\n\ndef dynamic_gate_pair(\n    X_gate_meta, y_meta, p_a_meta, p_b_meta,\n    X_gate_select, y_select, p_a_select, p_b_select,\n    X_gate_test, p_a_test, p_b_test,\n    w_min_opts, w_max_opts, recall_min\n):\n    target = (\n        np.abs(y_meta - p_b_meta)\n        < np.abs(y_meta - p_a_meta)\n    ).astype(int)\n\n    if len(np.unique(target)) < 2:\n        raw_select = np.full(len(y_select), target.mean())\n        raw_test = np.full(len(p_a_test), target.mean())\n        gate = None\n    else:\n        gate = Pipeline([\n            ("scaler", StandardScaler()),\n            ("clf", LogisticRegression(\n                max_iter=3000,\n                C=0.35,\n                class_weight="balanced",\n            )),\n        ])\n\n        gate.fit(X_gate_meta, target)\n\n        raw_select = gate.predict_proba(X_gate_select)[:, 1]\n        raw_test = gate.predict_proba(X_gate_test)[:, 1]\n\n    best = None\n\n    for w_min in w_min_opts:\n        for w_max in w_max_opts:\n            if w_max <= w_min:\n                continue\n\n            w = w_min + (w_max - w_min) * raw_select\n            p = w * p_b_select + (1.0 - w) * p_a_select\n\n            cand = tune_threshold(y_select, p, recall_min=recall_min)\n            cand["gate_w_min"] = float(w_min)\n            cand["gate_w_max"] = float(w_max)\n\n            if better(cand, best, recall_min):\n                best = cand\n\n    w_test = best["gate_w_min"] + (best["gate_w_max"] - best["gate_w_min"]) * raw_test\n    p_test = w_test * p_b_test + (1.0 - w_test) * p_a_test\n\n    return gate, best, p_test, w_test\n\n\ndef softmax_gate_3(\n    X_meta, y_meta, probs_meta,\n    X_select, y_select, probs_select,\n    X_test, probs_test,\n    names, recall_min\n):\n    P_meta = np.vstack([probs_meta[n] for n in names]).T\n    target = np.argmin(np.abs(P_meta - y_meta.reshape(-1, 1)), axis=1)\n\n    gate = Pipeline([\n        ("scaler", StandardScaler()),\n        ("clf", LogisticRegression(\n            max_iter=3000,\n            C=0.35,\n            multi_class="multinomial",\n            class_weight="balanced",\n        )),\n    ])\n\n    gate.fit(X_meta, target)\n\n    def aligned_predict(model, X):\n        raw = model.predict_proba(X)\n        classes = model.named_steps["clf"].classes_\n\n        out = np.zeros((len(X), len(names)))\n\n        for j, cls in enumerate(classes):\n            out[:, int(cls)] = raw[:, j]\n\n        s = out.sum(axis=1, keepdims=True)\n        s[s == 0] = 1.0\n\n        return out / s\n\n    W_select = aligned_predict(gate, X_select)\n    P_select = np.vstack([probs_select[n] for n in names]).T\n    p_select = (W_select * P_select).sum(axis=1)\n\n    thr = tune_threshold(y_select, p_select, recall_min=recall_min)\n\n    W_test = aligned_predict(gate, X_test)\n    P_test = np.vstack([probs_test[n] for n in names]).T\n    p_test = (W_test * P_test).sum(axis=1)\n\n    return gate, thr, p_test, W_test\n\n\ndef add_row(rows, model, role, threshold, ev, extra=None):\n    row = {\n        "model": model,\n        "role": role,\n        "best_threshold": threshold,\n        "weight_rf": np.nan,\n        "weight_xgb": np.nan,\n        "weight_tree": np.nan,\n        "weight_logistic": np.nan,\n        "gate_w_min": np.nan,\n        "gate_w_max": np.nan,\n    }\n\n    if extra:\n        row.update(extra)\n\n    row.update(ev)\n    rows.append(row)\n\n\ndef cross_error_analysis(y, pred_dict, out):\n    rows = []\n    names = list(pred_dict.keys())\n\n    for a in names:\n        for b in names:\n            if a == b:\n                continue\n\n            pa = pred_dict[a]\n            pb = pred_dict[b]\n\n            rows.append({\n                "base_model": a,\n                "helper_model": b,\n                "base_FN_helper_TP": int(((y == 1) & (pa == 0) & (pb == 1)).sum()),\n                "base_FP_helper_TN": int(((y == 0) & (pa == 1) & (pb == 0)).sum()),\n                "base_TP_helper_FN": int(((y == 1) & (pa == 1) & (pb == 0)).sum()),\n                "base_TN_helper_FP": int(((y == 0) & (pa == 0) & (pb == 1)).sum()),\n                "both_FN": int(((y == 1) & (pa == 0) & (pb == 0)).sum()),\n                "both_FP": int(((y == 0) & (pa == 1) & (pb == 1)).sum()),\n            })\n\n    pd.DataFrame(rows).to_csv(out, index=False)\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n\n    ap.add_argument("--data", required=True)\n    ap.add_argument("--out-dir", required=True)\n    ap.add_argument("--split-mode", default="group", choices=["group", "random"])\n    ap.add_argument("--split-manifest", default="", help="Shared split generated before feature selection.")\n    ap.add_argument("--n-jobs", type=int, default=max(1, min(4, os.cpu_count() or 2)))\n    ap.add_argument("--rf-trees", type=int, default=300)\n    ap.add_argument("--xgb-trees", type=int, default=450)\n    ap.add_argument("--xgb-device", default="cpu", choices=["cpu", "cuda", "gpu"])\n    ap.add_argument("--random-state", type=int, default=42)\n    ap.add_argument("--recall-min", type=float, default=0.95)\n    ap.add_argument("--weight-step", type=float, default=0.05)\n    ap.add_argument("--gate-w-min-options", default="0.00,0.02,0.05")\n    ap.add_argument("--gate-w-max-options", default="0.20,0.35,0.50,0.65")\n\n    args = ap.parse_args()\n\n    out_dir = Path(args.out_dir)\n    out_dir.mkdir(parents=True, exist_ok=True)\n\n    df, X, y, groups, gcol, const_cols = load_dataset(args.data)\n\n    if args.split_manifest:\n        try:\n            from .protocol import load_partitions\n        except ImportError:\n            from protocol import load_partitions\n        partitions = load_partitions(args.split_manifest, args.data, df)\n        train_idx, test_idx = partitions[\'train\'], partitions[\'test\']\n        val_idx = np.concatenate([partitions[\'meta\'], partitions[\'select\']])\n        split_used = "group_split_before_feature_selection_v1"\n    elif args.split_mode == "group":\n        train_idx, val_idx, test_idx, split_used = split_data(y, groups, args.random_state)\n    else:\n        idx = np.arange(len(y))\n\n        train_val_idx, test_idx = train_test_split(\n            idx,\n            test_size=0.20,\n            stratify=y,\n            random_state=args.random_state,\n        )\n\n        train_idx, val_idx = train_test_split(\n            train_val_idx,\n            test_size=0.25,\n            stratify=y[train_val_idx],\n            random_state=args.random_state,\n        )\n\n        split_used = "stratified_random"\n\n    if args.split_manifest:\n        meta_idx, select_idx = partitions[\'meta\'], partitions[\'select\']\n    else:\n        log("[WARN] Legacy split mode: feature selection must already exclude test data.")\n        meta_local, select_local = train_test_split(\n            np.arange(len(val_idx)), test_size=0.50, stratify=y[val_idx],\n            random_state=args.random_state + 77,\n        )\n        meta_idx = val_idx[meta_local]\n        select_idx = val_idx[select_local]\n\n    X_train = X.iloc[train_idx]\n    X_val = X.iloc[val_idx]\n    X_meta = X.iloc[meta_idx]\n    X_select = X.iloc[select_idx]\n    X_test = X.iloc[test_idx]\n\n    y_train = y[train_idx]\n    y_val = y[val_idx]\n    y_meta = y[meta_idx]\n    y_select = y[select_idx]\n    y_test = y[test_idx]\n\n    log("[3] Split train/val/test...")\n    log("[INFO] split used:", split_used)\n    log("[INFO] group col:", gcol)\n    log("[INFO] train:", X_train.shape, dict(pd.Series(y_train).value_counts().sort_index()))\n    log("[INFO] val:", X_val.shape, dict(pd.Series(y_val).value_counts().sort_index()))\n    log("[INFO] meta-val:", X_meta.shape, dict(pd.Series(y_meta).value_counts().sort_index()))\n    log("[INFO] select-val:", X_select.shape, dict(pd.Series(y_select).value_counts().sort_index()))\n    log("[INFO] test:", X_test.shape, dict(pd.Series(y_test).value_counts().sort_index()))\n\n    neg = max((y_train == 0).sum(), 1)\n    pos = max((y_train == 1).sum(), 1)\n    spw = neg / pos\n\n    log("[4] Build models...")\n    log("[INFO] scale_pos_weight:", spw)\n    log("[INFO] recall_min:", args.recall_min)\n\n    models = build_models(spw, args.n_jobs, args.random_state, args.rf_trees, args.xgb_trees, args.xgb_device)\n\n    probs_val = {}\n    probs_meta = {}\n    probs_select = {}\n    probs_test = {}\n    thresholds = {}\n    rows = []\n    base_pred_test = {}\n\n    log("[5] Train base models...")\n\n    for name, model in models.items():\n        log()\n        log("[RUN]", name)\n\n        model.fit(X_train, y_train)\n\n        probs_val[name] = prob(model, X_val)\n        probs_meta[name] = prob(model, X_meta)\n        probs_select[name] = prob(model, X_select)\n        probs_test[name] = prob(model, X_test)\n\n        thr = tune_threshold(y_val, probs_val[name])\n        thresholds[name] = thr["threshold"]\n\n        ev = eval_prob(y_test, probs_test[name], thr["threshold"])\n        add_row(rows, name, "base", thr["threshold"], ev)\n\n        base_pred_test[name] = (probs_test[name] >= thr["threshold"]).astype(int)\n\n        joblib.dump(model, out_dir / f"{name}.joblib")\n\n        log("[OK]", name)\n        log(" threshold:", thr["threshold"])\n        log(" test_f1_vulnerable:", ev["test_f1_vulnerable"])\n        log(" test_recall_vulnerable:", ev["test_recall_vulnerable"])\n        log(" confusion:", {\n            "tn": ev["test_tn"],\n            "fp": ev["test_fp"],\n            "fn": ev["test_fn"],\n            "tp": ev["test_tp"],\n        })\n\n    log()\n    log("[6] Weighted Soft Voting ensembles...")\n\n    ensemble_specs = OrderedDict({\n        "WEIGHT_RF_XGB": ["RandomForest", "XGBoost"],\n        "WEIGHT_RF_DT": ["RandomForest", "DecisionTree"],\n        "WEIGHT_XGB_DT": ["XGBoost", "DecisionTree"],\n        "WEIGHT_RF_XGB_DT": ["RandomForest", "XGBoost", "DecisionTree"],\n        "WEIGHT_ALL4": ["RandomForest", "XGBoost", "DecisionTree", "LogisticRegression"],\n    })\n\n    weighted_infos = {}\n\n    for ens_name, names in ensemble_specs.items():\n        info = optimize_weighted(\n            y_select,\n            probs_select,\n            names,\n            recall_min=args.recall_min,\n            step=args.weight_step,\n        )\n\n        weighted_infos[ens_name] = info\n\n        p_test = apply_weighted(probs_test, info)\n        ev = eval_prob(y_test, p_test, info["threshold"])\n\n        extra = {\n            "weight_rf": info["weights"].get("RandomForest", np.nan),\n            "weight_xgb": info["weights"].get("XGBoost", np.nan),\n            "weight_tree": info["weights"].get("DecisionTree", np.nan),\n            "weight_logistic": info["weights"].get("LogisticRegression", np.nan),\n        }\n\n        add_row(rows, ens_name, "weighted_soft_voting", info["threshold"], ev, extra)\n\n        log("[OK]", ens_name, info["weights"], "thr=", info["threshold"], "F1=", ev["test_f1_vulnerable"])\n\n    log()\n    log("[7] Stacking Logistic meta_model(...)")\n\n    base_names = ["RandomForest", "XGBoost", "DecisionTree", "LogisticRegression"]\n    risk_cols = select_risk_cols(X)\n\n    log("[INFO] selected risk/safe cols:", len(risk_cols))\n\n    X_meta_prob = meta_matrix(\n        X_meta,\n        {n: probs_meta[n] for n in base_names},\n        risk_cols=None,\n    )\n\n    X_select_prob = meta_matrix(\n        X_select,\n        {n: probs_select[n] for n in base_names},\n        risk_cols=None,\n    )\n\n    X_test_prob = meta_matrix(\n        X_test,\n        {n: probs_test[n] for n in base_names},\n        risk_cols=None,\n    )\n\n    stack_prob, thr_prob, p_stack_prob = train_stack(\n        X_meta_prob,\n        y_meta,\n        X_select_prob,\n        y_select,\n        X_test_prob,\n        args.recall_min,\n        C=0.50,\n    )\n\n    ev = eval_prob(y_test, p_stack_prob, thr_prob["threshold"])\n    add_row(rows, "STACK_LOGISTIC_PROB_ONLY", "stacking", thr_prob["threshold"], ev)\n    joblib.dump(stack_prob, out_dir / "STACK_LOGISTIC_PROB_ONLY.joblib")\n\n    log("[OK] STACK_LOGISTIC_PROB_ONLY", "F1=", ev["test_f1_vulnerable"])\n\n    X_meta_risk = meta_matrix(\n        X_meta,\n        {n: probs_meta[n] for n in base_names},\n        risk_cols=risk_cols,\n    )\n\n    X_select_risk = meta_matrix(\n        X_select,\n        {n: probs_select[n] for n in base_names},\n        risk_cols=risk_cols,\n    )\n\n    X_test_risk = meta_matrix(\n        X_test,\n        {n: probs_test[n] for n in base_names},\n        risk_cols=risk_cols,\n    )\n\n    stack_risk, thr_risk, p_stack_risk = train_stack(\n        X_meta_risk,\n        y_meta,\n        X_select_risk,\n        y_select,\n        X_test_risk,\n        args.recall_min,\n        C=0.35,\n    )\n\n    ev = eval_prob(y_test, p_stack_risk, thr_risk["threshold"])\n    add_row(rows, "STACK_LOGISTIC_PROB_RISK", "stacking", thr_risk["threshold"], ev)\n    joblib.dump(stack_risk, out_dir / "STACK_LOGISTIC_PROB_RISK.joblib")\n\n    log("[OK] STACK_LOGISTIC_PROB_RISK", "F1=", ev["test_f1_vulnerable"])\n\n    log()\n    log("[8] Dynamic Gate ensembles...")\n\n    w_min_opts = parse_float_list(args.gate_w_min_options)\n    w_max_opts = parse_float_list(args.gate_w_max_options)\n\n    gate_pairs = [\n        ("RandomForest", "XGBoost"),\n        ("RandomForest", "DecisionTree"),\n        ("XGBoost", "DecisionTree"),\n    ]\n\n    for a, b in gate_pairs:\n        Xm = meta_matrix(X_meta, {a: probs_meta[a], b: probs_meta[b]}, risk_cols)\n        Xs = meta_matrix(X_select, {a: probs_select[a], b: probs_select[b]}, risk_cols)\n        Xt = meta_matrix(X_test, {a: probs_test[a], b: probs_test[b]}, risk_cols)\n\n        gate, info, p_gate, w_gate = dynamic_gate_pair(\n            Xm, y_meta, probs_meta[a], probs_meta[b],\n            Xs, y_select, probs_select[a], probs_select[b],\n            Xt, probs_test[a], probs_test[b],\n            w_min_opts,\n            w_max_opts,\n            args.recall_min,\n        )\n\n        ev = eval_prob(y_test, p_gate, info["threshold"])\n\n        name = f"DYNAMIC_GATE_{a}_PLUS_{b}"\n\n        extra = {\n            "gate_w_min": info["gate_w_min"],\n            "gate_w_max": info["gate_w_max"],\n        }\n\n        add_row(rows, name, "dynamic_pair_gate", info["threshold"], ev, extra)\n\n        if gate is not None:\n            joblib.dump(gate, out_dir / f"{name}.joblib")\n\n        log("[OK]", name, "F1=", ev["test_f1_vulnerable"])\n\n    names3 = ["RandomForest", "XGBoost", "DecisionTree"]\n\n    Xm3 = meta_matrix(X_meta, {n: probs_meta[n] for n in names3}, risk_cols)\n    Xs3 = meta_matrix(X_select, {n: probs_select[n] for n in names3}, risk_cols)\n    Xt3 = meta_matrix(X_test, {n: probs_test[n] for n in names3}, risk_cols)\n\n    gate3, info3, p_gate3, W_gate3 = softmax_gate_3(\n        Xm3,\n        y_meta,\n        {n: probs_meta[n] for n in names3},\n        Xs3,\n        y_select,\n        {n: probs_select[n] for n in names3},\n        Xt3,\n        {n: probs_test[n] for n in names3},\n        names3,\n        args.recall_min,\n    )\n\n    ev = eval_prob(y_test, p_gate3, info3["threshold"])\n    add_row(rows, "SOFTMAX_GATE_RF_XGB_DT", "dynamic_three_gate", info3["threshold"], ev)\n    joblib.dump(gate3, out_dir / "SOFTMAX_GATE_RF_XGB_DT.joblib")\n\n    log("[OK] SOFTMAX_GATE_RF_XGB_DT", "F1=", ev["test_f1_vulnerable"])\n\n    log()\n    log("[9] Save outputs...")\n\n    metrics = pd.DataFrame(rows).sort_values(\n        ["test_f1_vulnerable", "test_recall_vulnerable"],\n        ascending=False,\n    )\n\n    metrics.to_csv(out_dir / "model_metrics.csv", index=False)\n    (out_dir / "feature_columns.json").write_text(\n        json.dumps(list(X.columns), indent=2), encoding="utf-8"\n    )\n\n    pred_cols = [\n        c for c in [\n            "binary", "binary_norm", "source_core", "build_config",\n            "json_file", "function", "name_norm", "entry",\n            "label", "merge_stage",\n        ]\n        if c in df.columns\n    ]\n\n    pred = df.iloc[test_idx][pred_cols].copy()\n    pred["y_true"] = y_test\n\n    for n in base_names:\n        pred[f"p_{n}"] = probs_test[n]\n\n    pred["p_WEIGHT_RF_XGB_DT"] = apply_weighted(probs_test, weighted_infos["WEIGHT_RF_XGB_DT"])\n    pred["p_WEIGHT_ALL4"] = apply_weighted(probs_test, weighted_infos["WEIGHT_ALL4"])\n    pred["p_STACK_LOGISTIC_PROB_ONLY"] = p_stack_prob\n    pred["p_STACK_LOGISTIC_PROB_RISK"] = p_stack_risk\n    pred["p_SOFTMAX_GATE_RF_XGB_DT"] = p_gate3\n\n    pred.to_csv(out_dir / "test_predictions.csv", index=False)\n\n    cross_error_analysis(\n        y_test,\n        base_pred_test,\n        out_dir / "model_cross_error_analysis.csv",\n    )\n\n    run_info = {\n        "random_state": args.random_state,\n        "split_manifest": args.split_manifest or None,\n        "parameters": vars(args),\n        "python_version": sys.version,\n        "package_versions": {name: importlib.metadata.version(name) for name in\n                             ["scikit-learn", "pandas", "numpy", "xgboost", "joblib"]},\n        "data": args.data,\n        "split_used": split_used,\n        "group_col": gcol,\n        "train_shape": list(X_train.shape),\n        "val_shape": list(X_val.shape),\n        "meta_val_shape": list(X_meta.shape),\n        "select_val_shape": list(X_select.shape),\n        "test_shape": list(X_test.shape),\n        "feature_count": int(X.shape[1]),\n        "risk_safe_meta_feature_count": len(risk_cols),\n        "recall_min": args.recall_min,\n        "weight_step": args.weight_step,\n    }\n\n    with open(out_dir / "run_info.json", "w", encoding="utf-8") as f:\n        json.dump(run_info, f, indent=2, ensure_ascii=False)\n\n    log()\n    log("[DONE] wrote:", out_dir)\n    log(metrics.to_string(index=False))\n\n\nif __name__ == "__main__":\n    main()\n', 'requirements-training.txt': '# Versions printed in the supplied Kaggle training log.\nscikit-learn==1.6.1\npandas==2.3.3\nxgboost==3.2.0\n# Exact historical versions were not supplied for these packages.\nnumpy>=1.26,<3\njoblib>=1.4,<2\nmatplotlib>=3.8,<4\n'}
for relative, content in BUNDLED_SOURCES.items():
    path = WORK / relative
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding='utf-8')
print('Prepared', len(BUNDLED_SOURCES), 'source/configuration files')


In [ ]:
INSTALL_DEPENDENCIES = False  # Bật nếu cần; việc tải package cần Internet.
if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(WORK / 'requirements-training.txt')], check=True)
for package in ['pandas', 'numpy', 'scikit-learn', 'xgboost', 'joblib', 'matplotlib']:
    print(package, importlib.metadata.version(package))
# Nếu vừa đổi phiên bản thư viện đã import trong kernel, khởi động lại session trước khi train.


In [ ]:
DATA_CSV = WORK / CSV_NAME
if INPUT_FILE is not None:
    candidates = [Path(INPUT_FILE)]
else:
    candidates = sorted(INPUT_ROOT.rglob(CSV_NAME))
if len(candidates) > 1:
    raise ValueError('Có nhiều CSV; đặt INPUT_FILE để chọn rõ dữ liệu.')
if candidates:
    shutil.copyfile(candidates[0], DATA_CSV)
else:
    archives = sorted(INPUT_ROOT.rglob('*.tar.gz'))
    matches = []
    for archive in archives:
        with tarfile.open(archive, 'r:gz') as handle:
            matches.extend((archive, member.name) for member in handle.getmembers()
                           if member.isfile() and Path(member.name).name == CSV_NAME)
    if len(matches) != 1:
        raise ValueError(f'Cần đúng một CSV trong input hoặc archive; tìm thấy {len(matches)}.')
    archive, member_name = matches[0]
    with tarfile.open(archive, 'r:gz') as handle:
        with handle.extractfile(member_name) as source, DATA_CSV.open('wb') as destination:
            shutil.copyfileobj(source, destination)
print('Input:', DATA_CSV, 'bytes:', DATA_CSV.stat().st_size)


## Huấn luyện

Cell này chạy selector RF/ExtraTrees, mutual information, correlation pruning, rồi bốn model cơ sở
và ensemble. CPU và tìm trọng số có thể tốn nhiều thời gian. Đây không phải cell kiểm tra nhanh.
Đặt `PREPARE_ONLY=True` để chỉ kiểm tra cách chia nhóm; chưa train hoặc chọn feature.


In [ ]:
PREPARE_ONLY = False
RUN_DIR = WORK / ('experiment_' + uuid4().hex[:8])
command = [sys.executable, str(WORK / 'training/run_experiment.py'),
           '--data', str(DATA_CSV), '--out-dir', str(RUN_DIR),
           '--features', str(FEATURE_COUNT), '--n-jobs', str(N_JOBS), '--seed', str(SEED)]
if PREPARE_ONLY:
    command.append('--prepare-only')
with (WORK / 'training.log').open('w', encoding='utf-8') as log:
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end='')
            log.write(line)
        returncode = process.wait()
    if returncode:
        raise RuntimeError(f'Training failed with exit code {returncode}; inspect training.log')


In [ ]:
import pandas as pd
metrics_file = RUN_DIR / 'models/model_metrics.csv'
if metrics_file.exists():
    display(pd.read_csv(metrics_file))
else:
    print('Chưa có metrics; PREPARE_ONLY chỉ tạo split manifest.')


## Xuất kết quả

ZIP chứa model, schema, metrics, tham số, phiên bản thư viện và split manifest. Dữ liệu đầy đủ và
test_predictions.csv không được đưa vào ZIP này. Model train 160 feature cần CSV/schema 160 tương ứng;
không thay riêng model vào bundle demo 260 feature.

Tập test vẫn có thể bị dùng gián tiếp nếu liên tục chọn cấu hình theo kết quả của nó. Hãy quyết định
thiết kế bằng validation và có thêm tập kiểm chứng cuối từ chương trình/nhóm mẫu chưa dùng.


In [ ]:
ARCHIVE = WORK / 'SBO_training_artifacts.zip'
with zipfile.ZipFile(ARCHIVE, 'w', zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
    for file in (RUN_DIR / 'models').glob('*'):
        if file.is_file() and file.name != 'test_predictions.csv':
            archive.write(file, 'models/sbo_detector/' + file.name)
    for filename in ['split_manifest.json', 'source_split_manifest.json']:
        file = RUN_DIR / filename
        if file.exists():
            archive.write(file, filename)
    archive.write(WORK / 'training.log', 'training.log')
print('Download:', ARCHIVE)
from IPython.display import FileLink
display(FileLink(str(ARCHIVE)))
